In [1]:
import os
from dotenv import load_dotenv

In [2]:
from langchain_deepseek import ChatDeepSeek
from langchain_core import tools
from langchain.agents import create_agent
from langchain_tavily import TavilySearch, TavilyExtract
from tavily import TavilyClient
from langchain.tools import tool
from langchain.agents.middleware import ModelRetryMiddleware
from langchain.agents.middleware import ModelCallLimitMiddleware

In [3]:
from libmumd import convert_file

In [4]:
load_dotenv()

True

In [5]:
import os

# Define your PDF directory and filename
pdf_directory = r"C:\path\to\your\lab_reports"  # <-- CHANGE THIS
pdf_filename = "sample_report.pdf"              # <-- CHANGE THIS
pdf_path = os.path.join(pdf_directory, pdf_filename)

# Convert PDF to markdown (handles RTL correctly)
result = convert_file(pdf_path, verbose=0)

if result['status'] == 'ok':
    # Read the generated markdown file
    md_path = result['output']
    with open(md_path, 'r', encoding='utf-8') as f:
        instructions = f.read()
else:
    raise RuntimeError(f"PDF conversion failed: {result}")

print(f"Extracted {len(instructions)} characters from PDF.")

ImportError: pymupdf4llm not installed. Run: pip install pymupdf4llm

In [6]:
sample_pdf_path = os.path.join(pdf_directory, "sample_expected_report.pdf")  # <-- CHANGE THIS

# Using libmumd
result_sample = convert_file(sample_pdf_path, verbose=0)
if result_sample['status'] == 'ok':
    with open(result_sample['output'], 'r', encoding='utf-8') as f:
        sample = f.read()

# Or using pypdf (same loop as above)

ImportError: pymupdf4llm not installed. Run: pip install pymupdf4llm

In [ ]:
todo_prompt = ...

In [7]:
import os

# Retrieve API key from environment (loaded via load_dotenv() in Cell 3)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY not found. Add it to your .env file.")

# --- Web Search Tool ---
# Useful for looking up references, standards, or background info for the lab report
tavily_search = TavilySearch(
    max_results=5,
    topic="general",          # or "news" if you need recent info
    include_answer=True,       # get a short synthesized answer
    include_raw_content=False, # keep payload light; use Extract if you need full text
    tavily_api_key=TAVILY_API_KEY,
)

# --- URL Content Extraction Tool ---
# Useful when the agent finds a promising link and needs the full page text
tavily_extract = TavilyExtract(
    extract_depth="basic",     # "advanced" is slower but parses more content
    tavily_api_key=TAVILY_API_KEY,
)

tools = [tavily_search, tavily_extract]

In [8]:
from langchain.agents.middleware import AgentMiddleware, hook_config
from langchain_core.messages import HumanMessage, AIMessage
from typing import Any

class LabReportValidatorMiddleware(AgentMiddleware):
    """
    After the model generates a response, check whether the Markdown content meets the lab report standards.
    If it does not meet the standards, inject a correction instruction and let the model regenerate.
    """
    def __init__(self, max_retries: int = 2):
        super().__init__()
        self.max_retries = max_retries

    @hook_config(can_jump_to=["model"])  # Allow jumping back to the model node to regenerate
    def after_model(self, state: dict[str, Any], runtime: Any) -> dict[str, Any] | None:
        messages = state["messages"]
        last_msg = messages[-1]

        # Only check the final text response (no tool_calls)
        if not isinstance(last_msg, AIMessage) or last_msg.tool_calls:
            return None

        content = last_msg.content or ""
        retry_count = state.get("_validation_retry", 0)

        # ---------- Your validation rules ----------
        issues = []

        # Rule 1: Must contain TODO placeholders marked with ==== (for images/charts you need to insert yourself)
        if "====" not in content:
            issues.append("Response does not contain any `====` placeholder markers, so it cannot guide the user to manually insert images or charts.")

        # Rule 2: Must contain at least one structured heading (Obsidian Markdown style)
        if not any(h in content for h in ["## ", "### "]):
            issues.append("Response lacks Markdown heading structure (## / ###), which does not meet Obsidian note formatting requirements.")

        # Rule 3 (optional): Must include a section referencing the dataset or raw data
        if "داده" not in content and "dataset" not in content.lower() and "دیتاست" not in content:
            issues.append("Response does not mention the dataset or raw data section, which does not meet the lab report completeness requirements.")

        # ---------- If there are issues and retry limit not exceeded, request regeneration ----------
        if issues and retry_count < self.max_retries:
            # Increment retry counter (stored in custom state field)
            state["_validation_retry"] = retry_count + 1

            # Append a HumanMessage to instruct the model to fix the issues
            correction = HumanMessage(
                content=(
                    "The response you just generated does not meet the lab report output standards. Issues are as follows:\n"
                    + "\n".join(f"- {i}" for i in issues)
                    + "\nPlease regenerate the complete Markdown response, ensuring all the above requirements are met. "
                      "Remember to mark parts that require manual insertion of images/charts with `====`."
                )
            )
            messages.append(correction)
            return {"messages": messages, "jump_to": "model"}

        # Passed or retry limit exceeded — clear flag and continue
        if "_validation_retry" in state:
            del state["_validation_retry"]
        return None

In [10]:
from langchain.agents.middleware import ModelRetryMiddleware, ModelCallLimitMiddleware

agent = create_agent(
    model=ChatDeepSeek(...),
    tools=tools,
    middleware=[
        ModelCallLimitMiddleware(run_limit=10),      # Prevent infinite loops
        ModelRetryMiddleware(max_retries=2),         # Network-level retry
        LabReportValidatorMiddleware(max_retries=2), # Content quality check
    ],
)

TypeError: BaseModel.__init__() takes 1 positional argument but 2 were given

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": todo_prompt}]
})
print(result["messages"][-1].content)